# B-field sweep for the 2D spin/exchange Thomas--Fermi solver

This notebook is the spin-resolved/exchange analogue of `sweeppone2.ipynb`.
It is designed to be compatible with the output of the previous notebook
`gpt_TF_confinement_B_sweep_fixed_dist_2D_half_top_gate_spin_exchange.ipynb`:
when resuming, it first looks for `U_scalar` in `data.npz`, and falls back to `U`
for older no-spin runs.

Model used in each B step:

\[
U_\sigma(x,y)=U_{\rm ext}(x,y)+U_H[n_\uparrow+n_\downarrow](x,y)+V_x[n_\sigma](x,y).
\]

The sweep saves every converged step in `RESULTS_DIR`, updates a `latest_simulation.json`
pointer, and appends the selected potential difference to `dU_spin_exchange.dat`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, replace
from pathlib import Path
import json, dataclasses, shutil

from scipy.special import erf
from scipy.optimize import brentq
from scipy.signal import fftconvolve
from scipy.ndimage import gaussian_filter
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import HTML, display

plt.rcParams.update({"font.size": 12})
import sys


In [ ]:
@dataclass
class Params2D:
    # full rectangular Hall bar dimensions in nm
    Lx: float = 1000.0
    Ly: float = 500.0
    Nx: int = 201
    Ny: int = 101

    use_elliptic_mask: bool = False

    # material / electrostatics
    eps_r: float = 13.0
    n_donor: float = 4.0e-3
    n_target: float = 2.0e-3
    donor_margin_nm: float = 0.0
    donor_smooth_nm: float = 0.0

    # top-gate distance in nm. Use None for bare Coulomb.
    dist_fixed: float | None = 200.0

    # Partial top-gate image configuration.
    # Images are included only for source charges in the selected x region.
    use_partial_top_gate: bool = True
    x_gate_edge_nm: float | None = 200.0

    # Donor image charges use a top-gate distance dist_fixed + donor_extra_dist_nm.
    donor_extra_dist_nm: float = -40.0

    # magnetic field / LLs
    B: float = 1.0
    g_spin: float = 0.0       # dimensionless old convention: E_Z = g_spin * hbar_omega_c
    n_LL: int = 12
    gamma_rel: float = 0.01
    use_B_scaling: bool = True

    # local spin exchange
    use_exchange: bool = True
    exchange_strength: float = 1.0
    exchange_prefactor: float = np.sqrt(np.pi/2.0)
    exchange_power: float = 0.5
    exchange_nu_clip: float = 1.0  # clip local spin filling in Vx to avoid runaway at very high density
    exchange_regularization: float = 1e-12

    # self-consistency
    max_iter: int = 10000
    mix: float = 0.002
    tol: float = 1e-8
    smooth_sigma_nm: float = 0.0

    # Coulomb regularization at r=0; if None, use area-equivalent disk radius
    diagonal_cutoff_nm: float | None = None

    @property
    def kappa0(self):
        # e^2/(4*pi*eps0*eps_r) in meV nm
        return 1440.0 / self.eps_r

    @property
    def hbar_omega_c(self):
        # GaAs-like value in meV/T times B
        return 1.76 * self.B

    @property
    def lB(self):
        return 25.658 / np.sqrt(self.B)

    @property
    def Gamma(self):
        if self.use_B_scaling:
            return self.gamma_rel * self.hbar_omega_c * np.sqrt(10.0 / self.B)
        return self.gamma_rel * self.hbar_omega_c

    @property
    def n_phi(self):
        # single-spin Landau degeneracy density in nm^-2
        return 1.0/(2*np.pi*self.lB**2)

p = Params2D()


In [ ]:

# --- Resume helpers ---------------------------------------------------------

def _json_safe_value(v):
    """Convert common numpy/path values to JSON-safe Python values."""
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    if isinstance(v, Path):
        return str(v)
    return v


def params_to_json_dict(p: Params2D):
    return {k: _json_safe_value(v) for k, v in dataclasses.asdict(p).items()}


def latest_saved_simulation_dir(results_dir):
    """Return the folder of the latest saved simulation, or None if not found."""
    latest_file = Path(results_dir) / "latest_simulation.json"
    if latest_file.exists():
        with open(latest_file, "r") as f:
            info = json.load(f)
        run_dir = Path(info["run_dir"])
        if not run_dir.is_absolute():
            run_dir = Path(results_dir) / run_dir
        if run_dir.exists():
            return run_dir

    if not Path(results_dir).exists():
        return None
    candidates = sorted([d for d in Path(results_dir).iterdir() if d.is_dir() and (d / "data.npz").exists()])
    return candidates[-1] if candidates else None


def load_saved_simulation(run_dir):
    """Load metadata and numerical arrays from a saved simulation directory."""
    run_dir = Path(run_dir)
    with open(run_dir / "metadata.json", "r") as f:
        metadata = json.load(f)
    data = np.load(run_dir / "data.npz", allow_pickle=False)
    return metadata, data


def load_params_and_initial_guess(
    p,
    results_dir,
    parameter_overrides=None,
    use_latest_saved_simulation=True,
    use_saved_u_as_initial_guess=True,
):
    """Load latest params and scalar potential, then apply parameter overrides.

    Compatibility logic:
    - new spin/exchange runs save `U_scalar`, `U_up`, `U_down`, and `U`;
    - older spinless runs save only `U`.

    For the spin/exchange solver the safest initial guess is the scalar electrostatic
    part. Therefore we use `U_scalar` when present and otherwise fall back to the
    older spinless `U`. All saved Params2D fields are restored before applying
    explicit sweep overrides such as the new B value.
    """
    parameter_overrides = {} if parameter_overrides is None else dict(parameter_overrides)
    resumed_run_dir = None
    resumed_metadata = None
    resumed_data = None
    U_initial = None

    if use_latest_saved_simulation:
        resumed_run_dir = latest_saved_simulation_dir(results_dir)
        if resumed_run_dir is None:
            print(f"No saved simulation found in {results_dir}. Starting from the current parameters.")
        else:
            resumed_metadata, resumed_data = load_saved_simulation(resumed_run_dir)
            saved_params = resumed_metadata.get("params", {})
            valid_fields = {f.name for f in dataclasses.fields(Params2D)}
            saved_params = {k: v for k, v in saved_params.items() if k in valid_fields}
            p = replace(p, **saved_params)
            print(f"Loaded parameters and data from: {resumed_run_dir}")

    if parameter_overrides:
        valid_fields = {f.name for f in dataclasses.fields(Params2D)}
        bad = sorted(set(parameter_overrides) - valid_fields)
        if bad:
            raise ValueError(f"Unknown Params2D fields in parameter_overrides: {bad}")
        p = replace(p, **parameter_overrides)
        print("Applied parameter overrides:", parameter_overrides)

    if resumed_data is not None and use_saved_u_as_initial_guess:
        saved_key = "U_scalar" if "U_scalar" in resumed_data.files else "U"
        saved_U = resumed_data[saved_key]
        if saved_U.shape == (p.Ny, p.Nx):
            U_initial = saved_U.copy()
            print(f"Using saved {saved_key} as scalar initial guess.")
        else:
            print(
                "Saved potential shape does not match the current grid; starting from U_ext. "
                f"saved_U.shape={saved_U.shape}, current={(p.Ny, p.Nx)}"
            )

    return p, U_initial, resumed_run_dir, resumed_metadata, resumed_data


In [ ]:

# --- Sweep controls ----------------------------------------------------------
# Magnetic-field sweep.
arr_B = np.linspace(1.0, 4.0, 300)

# Output of the diagnostic potential difference.
filename = "dU_spin_exchange.dat"

# Directory used both to read the latest output of the initialization notebook and
# to save the B-sweep steps. This default matches the spin/exchange initialization
# notebook output directory, so the first sweep step resumes from that simulation.
RESULTS_DIR = Path("results_2d_half_top_gate_spin_exchange")

# Start from the latest saved simulation in RESULTS_DIR.
# At the first step this is normally the initialization notebook output; after that
# it becomes the previous B-sweep point, because each step updates latest_simulation.json.
USE_LATEST_SAVED_SIMULATION = True
USE_SAVED_U_AS_INITIAL_GUESS = True

# Whether each B-step is saved as a full run.
SAVE_EACH_STEP = True

# B-independent overrides applied at every sweep step, after loading the latest run.
# Useful examples:
# BASE_PARAMETER_OVERRIDES = {
#     "mix": 5e-4,
#     "max_iter": 15000,
#     "use_exchange": True,
#     "exchange_strength": 1.0,
#     "g_spin": -0.44,
# }
BASE_PARAMETER_OVERRIDES = {}

# Points used to define dU = U(x_right,y_probe)-U(x_left,y_probe), after grounding.
x_right_probe_nm = 0.0
x_left_probe_nm = -350.0
y_probe_nm = 0.0

# Initial parameter object used only if no saved simulation is found.
p = Params2D()


## Electrostatics and spin/exchange solver definitions

In [ ]:
def make_grid_2d(p: Params2D):
    x = np.linspace(-0.5*p.Lx, 0.5*p.Lx, p.Nx)
    y = np.linspace(-0.5*p.Ly, 0.5*p.Ly, p.Ny)
    dx = x[1] - x[0]
    dy = y[1] - y[0]
    X, Y = np.meshgrid(x, y, indexing="xy")  # shape (Ny, Nx)
    return x, y, X, Y, dx, dy

def active_mask_2d(X, Y, p: Params2D):
    if not p.use_elliptic_mask:
        return np.ones_like(X, dtype=bool)
    return (X/(0.5*p.Lx))**2 + (Y/(0.5*p.Ly))**2 <= 1.0

def donor_density_profile_2d(X, Y, p: Params2D):
    mx = 0.5*p.Lx - p.donor_margin_nm
    my = 0.5*p.Ly - p.donor_margin_nm
    if p.donor_smooth_nm <= 0:
        mask = (np.abs(X) <= mx) & (np.abs(Y) <= my)
        return p.n_donor * mask.astype(float)
    s = p.donor_smooth_nm
    wx = 0.5*(np.tanh((X + mx)/s) - np.tanh((X - mx)/s))
    wy = 0.5*(np.tanh((Y + my)/s) - np.tanh((Y - my)/s))
    return p.n_donor * wx * wy

def source_mask_first_half_x_2d(X, p: Params2D, x_gate_edge_nm=None):
    """
    Source mask for the half-covered top gate.

    The partial-gate approximation is:
        image source exists only where source coordinate x' <= x_gate_edge_nm.

    This is not the exact Green function of a finite metallic gate, but it keeps
    the calculation FFT-fast and implements the intended source-side image model.
    """
    if x_gate_edge_nm is None:
        x_gate_edge_nm = p.x_gate_edge_nm
    if x_gate_edge_nm is None:
        x_gate_edge_nm = 0.5*(X.min() + X.max())
    return (X <= x_gate_edge_nm).astype(float)

def source_mask_first_half_x_2d_2(X, p: Params2D, x_gate_edge_nm=None):
    """
    Source mask for the half-covered top gate.
    Nonzero for
    x_gate_edge_nm - Lx <= x <= x_gate_edge_nm
    where Lx is inferred from the grid extent.
    Since the grid usually runs from -Lx/2 to +Lx/2, choosing
    x_gate_edge_nm = 0 gives the left half of the sample.
    """
    if x_gate_edge_nm is None:
        x_gate_edge_nm = p.x_gate_edge_nm
    if x_gate_edge_nm is None:
        x_gate_edge_nm = 0.5 * (X.min() + X.max())
    #x_left = X.min()+x_gate_edge_nm
    #x_right = X.max()-x_gate_edge_nm
    x_left = -x_gate_edge_nm
    x_right = x_gate_edge_nm
    return ((X <= x_left) | (X >= x_right)).astype(float)



def coulomb_kernel_2d_for_convolution(
    p: Params2D,
    dx,
    dy,
    z_source_nm=0.0,
    z_image_nm=None,
):
    """
    Open-boundary 2D Coulomb kernel for fftconvolve.

    Shape is (2*Ny-1, 2*Nx-1). Kernel units are 1/nm.

    If z_image_nm is None, this returns the direct kernel with vertical
    source separation z_source_nm. If z_image_nm is provided, it returns
    the image kernel with vertical separation z_image_nm.

    Multiplication by

        p.kappa0 * dx * dy * density * K

    gives meV, because p.kappa0 is e^2/(4*pi*eps0*eps_r) in meV nm and
    density is in nm^-2.
    """
    ix = np.arange(-(p.Nx-1), p.Nx)
    iy = np.arange(-(p.Ny-1), p.Ny)
    RX, RY = np.meshgrid(ix*dx, iy*dy, indexing="xy")
    rho2 = RX**2 + RY**2

    if p.diagonal_cutoff_nm is None:
        a = np.sqrt(dx*dy/np.pi)  # area-equivalent disk radius
    else:
        a = p.diagonal_cutoff_nm

    z = z_source_nm if z_image_nm is None else z_image_nm
    R = np.sqrt(rho2 + z*z)

    # Regularize only the true z=0 singularity.
    if abs(z) < 1e-15:
        R = np.maximum(R, a)

    return 1.0 / R

def convolve_open(field, K, dx, dy, p: Params2D):
    return dx*dy*fftconvolve(field, K, mode="same")

def make_kernel_and_background_2d(p: Params2D, dist=None):
    """
    Build grid, active mask, donor density, external donor potential, and
    a callable Hartree function.

    If p.use_partial_top_gate is True and dist is not None:
        - electron images exist only for electron source density in x <= x_gate_edge_nm;
        - donor images exist only for donor source density in x <= x_gate_edge_nm;
        - donor image charges use top-gate distance dist + p.donor_extra_dist_nm.

    Electron-electron Hartree:
        U_H = +kappa0 [ K_ee_direct * n - K_ee_image * (m_x n) ]

    Donor-electron external potential:
        U_donor = -kappa0 K_d_direct * nD + kappa0 K_d_image * (m_x nD)

    Donor image geometry:
        direct separation = 0
        image separation  = 2*(dist + donor_extra_dist_nm)
    """
    x, y, X, Y, dx, dy = make_grid_2d(p)
    mask = active_mask_2d(X, Y, p)
    nD = donor_density_profile_2d(X, Y, p)

    K_ee_direct = coulomb_kernel_2d_for_convolution(
        p, dx, dy,
        z_source_nm=0.0,
    )

    K_donor_direct = coulomb_kernel_2d_for_convolution(
        p, dx, dy,
        z_source_nm=0.0,
    )

    #top_gate_source_mask = source_mask_first_half_x_2d(X, p)
    top_gate_source_mask = source_mask_first_half_x_2d_2(X, p)

    if dist is None:
        K_ee_image = None
        K_donor_image = None

        U_donor = -p.kappa0 * convolve_open(nD, K_donor_direct, dx, dy, p)

        def hartree_from_density(n):
            return p.kappa0 * convolve_open(n, K_ee_direct, dx, dy, p)

        K_effective_for_diagnostics = K_ee_direct

    elif not p.use_partial_top_gate:
        K_ee_image = coulomb_kernel_2d_for_convolution(
            p, dx, dy,
            z_image_nm=2.0*dist,
        )

        K_donor_image = coulomb_kernel_2d_for_convolution(
            p, dx, dy,
            z_image_nm=2.0*(dist + p.donor_extra_dist_nm),
        )

        U_donor = (
            -p.kappa0 * convolve_open(nD, K_donor_direct, dx, dy, p)
            +p.kappa0 * convolve_open(nD, K_donor_image, dx, dy, p)
        )

        def hartree_from_density(n):
            return p.kappa0 * (
                convolve_open(n, K_ee_direct, dx, dy, p)
                - convolve_open(n, K_ee_image, dx, dy, p)
            )

        top_gate_source_mask = np.ones_like(X, dtype=float)
        K_effective_for_diagnostics = K_ee_direct - K_ee_image

    else:
        K_ee_image = coulomb_kernel_2d_for_convolution(
            p, dx, dy,
            z_image_nm=2.0*dist,
        )

        K_donor_image = coulomb_kernel_2d_for_convolution(
            p, dx, dy,
            z_image_nm=2.0*(dist + p.donor_extra_dist_nm),
        )

        U_donor = (
            -p.kappa0 * convolve_open(nD, K_donor_direct, dx, dy, p)
            +p.kappa0 * convolve_open(top_gate_source_mask*nD, K_donor_image, dx, dy, p)
        )

        def hartree_from_density(n):
            return p.kappa0 * (
                convolve_open(n, K_ee_direct, dx, dy, p)
                - convolve_open(top_gate_source_mask*n, K_ee_image, dx, dy, p)
            )

        # This is only diagnostic because the masked image term is not truly
        # represented by a translation-invariant effective kernel.
        K_effective_for_diagnostics = K_ee_direct - K_ee_image

    U_ext = U_donor.copy()
    U_ext -= np.nanmin(U_ext[mask])

    return (
        x, y, X, Y, dx, dy,
        mask,
        K_effective_for_diagnostics,
        nD,
        U_ext,
        top_gate_source_mask,
        hartree_from_density,
        K_ee_direct,
        K_ee_image,
        K_donor_direct,
        K_donor_image,
    )

(
    x, y, X, Y, dx, dy,
    mask,
    K,
    nD,
    U_ext,
    top_gate_source_mask,
    hartree_from_density,
    K_ee_direct,
    K_ee_image,
    K_donor_direct,
    K_donor_image,
) = make_kernel_and_background_2d(p, dist=p.dist_fixed)

fig1, ax = plt.subplots(figsize=(5.2, 4.2))
im = ax.imshow(U_ext-np.amin(U_ext), origin="lower", extent=[x.min(), x.max(), y.min(), y.max()], aspect="equal", vmin=0, vmax=20)
ax.set_xlabel("x [nm]"); ax.set_ylabel("y [nm]")
ax.set_title(r"$U_{donor}$ [meV], half top-gate image")
ax.grid()
fig1.colorbar(im, ax=ax)
fig1.tight_layout()

fig2, ax = plt.subplots(figsize=(5.2, 4.2))
im = ax.imshow(top_gate_source_mask, origin="lower", extent=[x.min(), x.max(), y.min(), y.max()], aspect="equal")
ax.set_xlabel("x [nm]"); ax.set_ylabel("y [nm]")
ax.set_title("Source region with top-gate images")
fig2.colorbar(im, ax=ax)
fig2.tight_layout()

In [ ]:
def spin_landau_energies(p: Params2D):
    """Return separate arrays for the two spin branches.

    Convention kept compatible with the older notebook:
        E_Z = g_spin * hbar_omega_c.
    The labels up/down are therefore branch labels; for negative g_spin their
    energetic ordering is reversed automatically.
    """
    n = np.arange(p.n_LL, dtype=float)
    E0 = p.hbar_omega_c*(n + 0.5)
    EZ = p.g_spin*p.hbar_omega_c
    E_up = E0 - 0.5*EZ
    E_down = E0 + 0.5*EZ
    return E_up, E_down


def density_spin_from_potentials_2d(U_up, U_down, mu, p: Params2D, mask=None):
    E_up, E_down = spin_landau_energies(p)
    arg_up = (mu - U_up[..., None] - E_up[None, None, :])/(np.sqrt(2.0)*p.Gamma)
    arg_down = (mu - U_down[..., None] - E_down[None, None, :])/(np.sqrt(2.0)*p.Gamma)
    occ_up = 0.5*(1.0 + erf(arg_up))
    occ_down = 0.5*(1.0 + erf(arg_down))
    n_up = p.n_phi*np.sum(occ_up, axis=-1)
    n_down = p.n_phi*np.sum(occ_down, axis=-1)
    if mask is not None:
        n_up = np.where(mask, n_up, 0.0)
        n_down = np.where(mask, n_down, 0.0)
    return n_up, n_down


def density_from_potential_2d(U, mu, p: Params2D, mask=None):
    """Compatibility helper: spin-resolved density without exchange splitting."""
    n_up, n_down = density_spin_from_potentials_2d(U, U, mu, p, mask=mask)
    return n_up + n_down


def compressibility_spin_from_potentials_2d(U_up, U_down, mu, p: Params2D, mask=None):
    E_up, E_down = spin_landau_energies(p)
    z_up = (mu - U_up[..., None] - E_up[None, None, :])/p.Gamma
    z_down = (mu - U_down[..., None] - E_down[None, None, :])/p.Gamma
    gaussian_up = np.exp(-0.5*z_up*z_up)/(np.sqrt(2*np.pi)*p.Gamma)
    gaussian_down = np.exp(-0.5*z_down*z_down)/(np.sqrt(2*np.pi)*p.Gamma)
    comp_up = p.n_phi*np.sum(gaussian_up, axis=-1)
    comp_down = p.n_phi*np.sum(gaussian_down, axis=-1)
    if mask is not None:
        comp_up = np.where(mask, comp_up, 0.0)
        comp_down = np.where(mask, comp_down, 0.0)
    return comp_up, comp_down


def compressibility_from_potential_2d(U, mu, p: Params2D, mask=None):
    comp_up, comp_down = compressibility_spin_from_potentials_2d(U, U, mu, p, mask=mask)
    return comp_up + comp_down


def local_exchange_potential_2d(n_sigma, p: Params2D, mask=None):
    """Local spin-dependent exchange potential in meV.

    Vx_sigma = -lambda_x C_x kappa0/lB * (n_sigma/n_phi)^alpha.
    The filling entering Vx is clipped by default to keep the approximation local
    to a single spin-resolved LL and avoid numerical runaway in high-density spots.
    """
    if (not p.use_exchange) or p.exchange_strength == 0.0:
        return np.zeros_like(n_sigma)
    nu_sigma = n_sigma/(p.n_phi + p.exchange_regularization)
    if p.exchange_nu_clip is not None and p.exchange_nu_clip > 0:
        nu_sigma = np.clip(nu_sigma, 0.0, p.exchange_nu_clip)
    else:
        nu_sigma = np.maximum(nu_sigma, 0.0)
    Vx = -p.exchange_strength*p.exchange_prefactor*(p.kappa0/p.lB)*np.power(nu_sigma, p.exchange_power)
    if mask is not None:
        Vx = np.where(mask, Vx, 0.0)
    return Vx


def filling_from_density_2d(n, p: Params2D):
    return n / p.n_phi


def spin_polarization_2d(n_up, n_down):
    n = n_up + n_down
    return np.divide(n_up - n_down, n, out=np.zeros_like(n), where=n > 0)


In [ ]:
def active_area(mask, dx, dy):
    return dx*dy*np.sum(mask)

def average_density_2d(n, mask, dx, dy):
    return dx*dy*np.sum(n[mask]) / active_area(mask, dx, dy)

def find_mu_for_target_spin_2d(U_up, U_down, p: Params2D, mask, dx, dy, verbose=False):
    E_up, E_down = spin_landau_energies(p)
    U_min = min(np.min(U_up[mask]), np.min(U_down[mask]))
    U_max = max(np.max(U_up[mask]), np.max(U_down[mask]))
    E_min = min(np.min(E_up), np.min(E_down))
    E_max = max(np.max(E_up), np.max(E_down))
    lo = U_min + E_min - 20*p.Gamma - 5*p.hbar_omega_c - abs(p.kappa0/p.lB)
    hi = U_max + E_max + 20*p.Gamma + 5*p.hbar_omega_c + abs(p.kappa0/p.lB)

    def f(mu):
        n_up, n_down = density_spin_from_potentials_2d(U_up, U_down, mu, p, mask=mask)
        return average_density_2d(n_up + n_down, mask, dx, dy) - p.n_target

    flo, fhi = f(lo), f(hi)
    if verbose:
        print("mu bracket", lo, hi, flo, fhi)
    if flo > 0 or fhi < 0:
        raise RuntimeError(
            f"Cannot bracket mu: f(lo)={flo:g}, f(hi)={fhi:g}. Increase n_LL or check target density."
        )
    return brentq(f, lo, hi, xtol=1e-12, rtol=1e-12, maxiter=100)

# Compatibility wrapper used by old helper cells.
def find_mu_for_target_2d(U, p: Params2D, mask, dx, dy, verbose=False):
    return find_mu_for_target_spin_2d(U, U, p, mask, dx, dy, verbose=verbose)


In [ ]:
def gaussian_smooth_2d(U, sigma_nm, dx, dy):
    if sigma_nm <= 0:
        return U
    sigma_pix = (sigma_nm/dy, sigma_nm/dx)
    return gaussian_filter(U, sigma=sigma_pix, mode="nearest")

def solve_self_consistent_2d_spin_exchange(p: Params2D, dist=None, U_initial=None, verbose=False):
    (
        x, y, X, Y, dx, dy,
        mask,
        K,
        nD,
        U_ext,
        top_gate_source_mask,
        hartree_from_density,
        K_ee_direct,
        K_ee_image,
        K_donor_direct,
        K_donor_image,
    ) = make_kernel_and_background_2d(p, dist=dist)

    U_H = np.zeros_like(U_ext)
    if U_initial is None:
        U_scalar = U_ext.copy()
    else:
        U_scalar = U_initial.copy()
    U_scalar -= np.nanmin(U_scalar[mask])

    Vx_up = np.zeros_like(U_scalar)
    Vx_down = np.zeros_like(U_scalar)
    U_up = U_scalar + Vx_up
    U_down = U_scalar + Vx_down

    mu = find_mu_for_target_spin_2d(U_up, U_down, p, mask, dx, dy)
    n_up, n_down = density_spin_from_potentials_2d(U_up, U_down, mu, p, mask=mask)
    n = n_up + n_down

    history = []
    for it in range(p.max_iter):
        U_H_new = hartree_from_density(n)
        Vx_up_new = local_exchange_potential_2d(n_up, p, mask=mask)
        Vx_down_new = local_exchange_potential_2d(n_down, p, mask=mask)

        U_scalar_new = U_ext + U_H_new
        U_scalar_new = gaussian_smooth_2d(U_scalar_new, p.smooth_sigma_nm, dx, dy)
        U_scalar_new -= np.nanmin(U_scalar_new[mask])

        # Mix Hartree/background and exchange components separately.
        U_scalar_mixed = (1.0 - p.mix)*U_scalar + p.mix*U_scalar_new
        Vx_up_mixed = (1.0 - p.mix)*Vx_up + p.mix*Vx_up_new
        Vx_down_mixed = (1.0 - p.mix)*Vx_down + p.mix*Vx_down_new

        U_up_mixed = U_scalar_mixed + Vx_up_mixed
        U_down_mixed = U_scalar_mixed + Vx_down_mixed
        outside_value = np.nanmax(U_scalar_mixed[mask])
        U_up_mixed = np.where(mask, U_up_mixed, outside_value)
        U_down_mixed = np.where(mask, U_down_mixed, outside_value)

        mu_new = find_mu_for_target_spin_2d(U_up_mixed, U_down_mixed, p, mask, dx, dy)
        n_up_new, n_down_new = density_spin_from_potentials_2d(U_up_mixed, U_down_mixed, mu_new, p, mask=mask)
        n_new = n_up_new + n_down_new

        dn = np.sqrt(np.mean((n_new[mask] - n[mask])**2))
        dpol = np.sqrt(np.mean(((n_up_new - n_down_new)[mask] - (n_up - n_down)[mask])**2))
        if it % 25 == 0 or it == p.max_iter - 1:
            n_avg = average_density_2d(n_new, mask, dx, dy)
            pol_avg = average_density_2d(n_up_new - n_down_new, mask, dx, dy)/max(n_avg, 1e-30)
            history.append((it, mu_new, n_avg, dn, dpol, pol_avg))
            if verbose:
                print(f"it={it:4d}, mu={mu_new:.6g}, n_avg={n_avg:.6g}, dn={dn:.3e}, dpol={dpol:.3e}, P_avg={pol_avg:.3g}")

        U_scalar, Vx_up, Vx_down = U_scalar_mixed, Vx_up_mixed, Vx_down_mixed
        U_up, U_down = U_up_mixed, U_down_mixed
        mu = mu_new
        n_up, n_down, n = n_up_new, n_down_new, n_new
        U_H = U_H_new

        if dn < p.tol and dpol < p.tol:
            history.append((it, mu, average_density_2d(n, mask, dx, dy), dn, dpol,
                            average_density_2d(n_up - n_down, mask, dx, dy)/max(average_density_2d(n, mask, dx, dy), 1e-30)))
            break

    comp_up, comp_down = compressibility_spin_from_potentials_2d(U_up, U_down, mu, p, mask=mask)
    return {
        "x": x, "y": y, "X": X, "Y": Y,
        "dx": dx, "dy": dy, "mask": mask,
        "dist": dist, "K": K,
        "K_ee_direct": K_ee_direct,
        "K_ee_image": K_ee_image,
        "K_donor_direct": K_donor_direct,
        "K_donor_image": K_donor_image,
        "top_gate_source_mask": top_gate_source_mask,
        "n_donor_profile": nD,
        "U_ext": U_ext,
        "U_H": U_H,
        "Vx_up": Vx_up,
        "Vx_down": Vx_down,
        "U_scalar": U_scalar,
        "U_up": U_up,
        "U_down": U_down,
        "U": 0.5*(U_up + U_down),
        "mu": mu,
        "n_up": n_up,
        "n_down": n_down,
        "n": n,
        "nu_up": filling_from_density_2d(n_up, p),
        "nu_down": filling_from_density_2d(n_down, p),
        "nu": filling_from_density_2d(n, p),
        "spin_polarization": spin_polarization_2d(n_up, n_down),
        "compressibility_up": comp_up,
        "compressibility_down": comp_down,
        "compressibility": comp_up + comp_down,
        "history": np.array(history),
    }


In [ ]:
from datetime import datetime


def save_simulation_2d_spin_exchange(res, figures=(), p: Params2D=None, results_dir=None, run_name=None, extra_metadata=None):
    if p is None:
        raise ValueError("p must be provided")
    if results_dir is None:
        results_dir = Path("sweep_fold_spin_exchange")
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    if run_name is None:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_name = f"sim_spin_x_B_{p.B:.5g}_dist_{p.dist_fixed}_xedge_{p.x_gate_edge_nm}_{stamp}"

    run_dir = results_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=False)

    data_path = run_dir / "data.npz"
    meta_path = run_dir / "metadata.json"

    arrays_to_save = {
        "x": res["x"], "y": res["y"], "X": res["X"], "Y": res["Y"],
        "mask": res["mask"],
        "n": res["n"], "n_up": res["n_up"], "n_down": res["n_down"],
        "nu": res["nu"], "nu_up": res["nu_up"], "nu_down": res["nu_down"],
        "spin_polarization": res["spin_polarization"],
        "U": res["U"], "U_scalar": res["U_scalar"], "U_up": res["U_up"], "U_down": res["U_down"],
        "U_ext": res["U_ext"], "U_H": res["U_H"], "Vx_up": res["Vx_up"], "Vx_down": res["Vx_down"],
        "compressibility": res["compressibility"],
        "compressibility_up": res["compressibility_up"], "compressibility_down": res["compressibility_down"],
        "n_donor_profile": res["n_donor_profile"],
        "top_gate_source_mask": res["top_gate_source_mask"],
        "K": res["K"],
        "K_ee_direct": res["K_ee_direct"],
        "K_donor_direct": res["K_donor_direct"],
        "history": res["history"],
    }
    if res["K_ee_image"] is not None:
        arrays_to_save["K_ee_image"] = res["K_ee_image"]
    if res["K_donor_image"] is not None:
        arrays_to_save["K_donor_image"] = res["K_donor_image"]

    np.savez_compressed(data_path, **arrays_to_save)

    avg_n = average_density_2d(res["n"], res["mask"], res["dx"], res["dy"])
    avg_pol = average_density_2d(res["n_up"] - res["n_down"], res["mask"], res["dx"], res["dy"])/max(avg_n, 1e-30)
    summary = {
        "mu": _json_safe_value(res["mu"]),
        "dist": _json_safe_value(res["dist"]),
        "dx": _json_safe_value(res["dx"]),
        "dy": _json_safe_value(res["dy"]),
        "average_density": _json_safe_value(avg_n),
        "average_spin_polarization": _json_safe_value(avg_pol),
        "target_density": _json_safe_value(p.n_target),
        "center_filling": _json_safe_value(res["nu"][p.Ny//2, p.Nx//2]),
        "center_filling_up": _json_safe_value(res["nu_up"][p.Ny//2, p.Nx//2]),
        "center_filling_down": _json_safe_value(res["nu_down"][p.Ny//2, p.Nx//2]),
        "center_Vx_up_meV": _json_safe_value(res["Vx_up"][p.Ny//2, p.Nx//2]),
        "center_Vx_down_meV": _json_safe_value(res["Vx_down"][p.Ny//2, p.Nx//2]),
        "nu_min": _json_safe_value(np.nanmin(res["nu"])),
        "nu_max": _json_safe_value(np.nanmax(res["nu"])),
        "converged_iterations_recorded": int(res["history"][-1, 0]) if len(res["history"]) else None,
        "final_dn": _json_safe_value(res["history"][-1, 3]) if len(res["history"]) else None,
        "final_dpol": _json_safe_value(res["history"][-1, 4]) if len(res["history"]) else None,
    }

    metadata = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "params": params_to_json_dict(p),
        "summary": summary,
        "resumed_from": None if extra_metadata is None else extra_metadata.get("resumed_from"),
        "parameter_overrides": {},
        "model_note": "Spin-resolved 2D Thomas-Fermi with local Vx_sigma[n_sigma].",
    }
    if extra_metadata is not None:
        metadata["extra_metadata"] = extra_metadata

    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    latest_path = results_dir / "latest_simulation.json"
    with open(latest_path, "w") as f:
        json.dump({"run_dir": run_dir.name, "created_at": metadata["created_at"]}, f, indent=2)

    names = [
        "Scalar_potential.png",
        "Electron_density.png",
        "Local_total_filling.png",
        "Local_compressibility.png",
        "Spin_polarization.png",
        "Exchange_spin_splitting.png",
        "Filling_spin_cuts.png",
    ]
    for fig, name in zip(figures, names):
        fig.savefig(run_dir / name, dpi=300)

    print(f"Saved simulation to: {run_dir}")
    print(f"  data:     {data_path}")
    print(f"  metadata: {meta_path}")
    print(f"Updated latest pointer: {latest_path}")
    return run_dir


## Run the B sweep

In [ ]:

dU = []
summary_rows = []

for nB, Bs in enumerate(arr_B):
    print("\n" + "="*80)
    print(f"B step {nB+1}/{len(arr_B)}: B = {Bs:.8g} T")

    parameter_overrides = dict(BASE_PARAMETER_OVERRIDES)
    parameter_overrides["B"] = float(Bs)

    p, U_INITIAL_FROM_SAVED_RUN, RESUMED_RUN_DIR, RESUMED_METADATA, RESUMED_DATA = load_params_and_initial_guess(
        p,
        results_dir=RESULTS_DIR,
        parameter_overrides=parameter_overrides,
        use_latest_saved_simulation=USE_LATEST_SAVED_SIMULATION,
        use_saved_u_as_initial_guess=USE_SAVED_U_AS_INITIAL_GUESS,
    )

    print("Active parameters:")
    print(p)
    print(f"dx, dy = {p.Lx/(p.Nx-1):.3f} nm, {p.Ly/(p.Ny-1):.3f} nm")
    print(f"grid points = {p.Nx*p.Ny}")
    print(f"lB = {p.lB:.3f} nm, hbar omega_c = {p.hbar_omega_c:.3f} meV, Gamma = {p.Gamma:.4f} meV")
    print(f"Expected average filling = {2*np.pi*p.lB**2*p.n_target:.4f}")

    res = solve_self_consistent_2d_spin_exchange(
        p,
        dist=p.dist_fixed,
        U_initial=U_INITIAL_FROM_SAVED_RUN,
        verbose=True,
    )

    avg_n = average_density_2d(res["n"], res["mask"], res["dx"], res["dy"])
    center = (p.Ny//2, p.Nx//2)
    print("history tail columns: it, mu, n_avg, dn, dpol, P_avg")
    print(res["history"][-5:])
    print("average density:", avg_n, "target:", p.n_target)
    print("center filling total/up/down:", res["nu"][center], res["nu_up"][center], res["nu_down"][center])

    if SAVE_EACH_STEP:
        run_name = f"sweep_spin_x_B_{p.B:.8g}_step_{nB:04d}"
        SAVED_RUN_DIR = save_simulation_2d_spin_exchange(
            res,
            figures=(),
            p=p,
            results_dir=RESULTS_DIR,
            run_name=run_name,
            extra_metadata={
                "sweep_index": int(nB),
                "sweep_B": float(Bs),
                "resumed_from": str(RESUMED_RUN_DIR) if RESUMED_RUN_DIR is not None else None,
                "base_parameter_overrides": {k: _json_safe_value(v) for k, v in BASE_PARAMETER_OVERRIDES.items()},
            },
        )
    else:
        SAVED_RUN_DIR = None

    x, y = res["x"], res["y"]
    # Use the scalar electrostatic potential for the sweep diagnostic, i.e. before adding Vx_sigma.
    # This is closest to the old spinless `U` diagnostic. For spin-dependent diagnostics, use
    # res["U_up"] or res["U_down"] instead.
    U_diag = res["U_scalar"]
    U_grounded = U_diag - np.nanmin(U_diag[res["mask"]])
    nx_right = np.argmin(np.abs(x - x_right_probe_nm))
    nx_left = np.argmin(np.abs(x - x_left_probe_nm))
    ny = np.argmin(np.abs(y - y_probe_nm))
    dU0 = U_grounded[ny, nx_right] - U_grounded[ny, nx_left]
    dU.append(dU0)

    row = {
        "B": float(Bs),
        "dU_scalar": float(dU0),
        "mu": float(res["mu"]),
        "avg_density": float(avg_n),
        "center_nu": float(res["nu"][center]),
        "center_nu_up": float(res["nu_up"][center]),
        "center_nu_down": float(res["nu_down"][center]),
        "center_spin_polarization": float(res["spin_polarization"][center]),
        "center_Vx_up": float(res["Vx_up"][center]),
        "center_Vx_down": float(res["Vx_down"][center]),
        "saved_run_dir": str(SAVED_RUN_DIR) if SAVED_RUN_DIR is not None else "",
    }
    summary_rows.append(row)

    mode = "w" if nB == 0 else "a"
    with open(filename, mode) as f:
        f.write(
            f"{row['B']}\t{row['dU_scalar']}\t{row['mu']}\t{row['center_nu']}\t"
            f"{row['center_nu_up']}\t{row['center_nu_down']}\t{row['center_spin_polarization']}\t"
            f"{row['center_Vx_up']}\t{row['center_Vx_down']}\n"
        )

    # The just-saved result becomes the natural initial guess for the next B step.
    USE_LATEST_SAVED_SIMULATION = SAVE_EACH_STEP

print("\nSweep complete.")
dU = np.asarray(dU)


In [ ]:

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(arr_B[:len(dU)], dU, marker="o")
ax.set_xlabel("B [T]")
ax.set_ylabel(r"$\Delta U_{\rm scalar}$ [meV]")
ax.set_title("Spin/exchange B sweep diagnostic")
fig.tight_layout()

# Also save a compact tabular summary.
try:
    import pandas as pd
    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv("sweep_spin_exchange_summary.csv", index=False)
    display(df_summary.head())
except Exception as exc:
    print("Could not create pandas summary:", exc)
